In [1]:
# Imports / global contants

# csv Dateien sind im Verzeichnis ../data zu finden

import pandas as pd
import glob
import os
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
from matplotlib import figure
import math
import numpy as np
import seaborn as sns
from scipy.misc import electrocardiogram
from scipy.signal import find_peaks
import gc
import pylab

import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio

from pandas.plotting import parallel_coordinates

%matplotlib inline

timeFormat = "%Y-%m-%dT%H:%M:%S.%fZ"

export = "../export/"
export_img = "../export/img/"
export_interactions = "../export/interaction/"

path = "../data/"

gtv_colormap = {1: 'rgb(54, 161, 185)', 2: 'rgb(0, 44, 61)', 3: 'rgb(233, 71, 74)'}
plot_bgcolor = 'rgba(239, 240, 231,.7)'

In [12]:
df_cleaned = pd.read_csv(rf'{export}cleanedStudy_interactionOnly.csv', sep=";")

In [42]:
# Beispiel-Daten erzeugen
np.random.seed(0)
zeiten = pd.date_range(start="2024-01-01", periods=200, freq="T")
werte = np.concatenate([
    np.zeros(60),              # 60 Nullwerte (vorzeitiger Start)
    np.random.rand(10),        # zufällige "Störung"
    np.zeros(45),              # noch eine Nullsequenz (zu kurz)
    np.random.normal(5, 1, 85) # echte Messung
])

df = pd.DataFrame({"zeit": zeiten, "wert": werte})
df["zeit"] = pd.to_datetime(df["zeit"])
df.set_index("zeit", inplace=True)

# Schwellenwert für "Null" (z.B. ≤ 0.1)
is_zero_like = df["wert"] <= 0.1

display(is_zero_like)

display(is_zero_like.loc[lambda x: x == True])

# Glättung: kurze Unterbrechungen ignorieren
smoothed = is_zero_like.rolling(window=5, center=True, min_periods=1).mean() > 0.5

# Gruppieren: gleiche True/False Blöcke finden
group_id = (smoothed != smoothed.shift()).cumsum()

display(group_id)

groups = df.groupby(group_id)

# Längere Nullblöcke identifizieren
to_remove = []
removed_durations = []

for _, group in groups:
    if (smoothed.loc[group.index[0]]):  # Nur Null-ähnliche Gruppen
        if len(group) > 50:
            to_remove.append(group.index)
            duration = group.index[-1] - group.index[0]
            removed_durations.append(duration)

# Indexes zusammenfassen und entfernen
remove_idx = pd.Index(np.concatenate(to_remove)) if to_remove else pd.Index([])
df_cleaned = df.drop(index=remove_idx)

# Ursprüngliche vs. korrigierte Dauer
original_duration = df.index[-1] - df.index[0]
corrected_duration = original_duration - sum(removed_durations, pd.Timedelta(0))

print("Ursprüngliche Dauer:", original_duration)
print("Dauer der entfernten Nullphasen:", sum(removed_durations, pd.Timedelta(0)))
print("Korrigierte Dauer:", corrected_duration)

zeit
2024-01-01 00:00:00     True
2024-01-01 00:01:00     True
2024-01-01 00:02:00     True
2024-01-01 00:03:00     True
2024-01-01 00:04:00     True
                       ...  
2024-01-01 03:15:00    False
2024-01-01 03:16:00    False
2024-01-01 03:17:00    False
2024-01-01 03:18:00    False
2024-01-01 03:19:00    False
Name: wert, Length: 200, dtype: bool

zeit
2024-01-01 00:00:00    True
2024-01-01 00:01:00    True
2024-01-01 00:02:00    True
2024-01-01 00:03:00    True
2024-01-01 00:04:00    True
                       ... 
2024-01-01 01:50:00    True
2024-01-01 01:51:00    True
2024-01-01 01:52:00    True
2024-01-01 01:53:00    True
2024-01-01 01:54:00    True
Name: wert, Length: 105, dtype: bool

zeit
2024-01-01 00:00:00    1
2024-01-01 00:01:00    1
2024-01-01 00:02:00    1
2024-01-01 00:03:00    1
2024-01-01 00:04:00    1
                      ..
2024-01-01 03:15:00    4
2024-01-01 03:16:00    4
2024-01-01 03:17:00    4
2024-01-01 03:18:00    4
2024-01-01 03:19:00    4
Name: wert, Length: 200, dtype: int32

Ursprüngliche Dauer: 0 days 03:19:00
Dauer der entfernten Nullphasen: 0 days 00:59:00
Korrigierte Dauer: 0 days 02:20:00


In [56]:
df_cleaned = pd.read_csv(rf'{export}cleanedStudy_interactionOnly.csv', sep=";")

proband = 18
block = 2
task = 4
trial = 3

display(df_cleaned)

exp_data = df_cleaned[(df_cleaned["Proband"] == proband) & (df_cleaned["Task"] == task) & (df_cleaned["Block"] == block) & (df_cleaned["Trial"] == trial)].reset_index(drop=True)

display(exp_data)

df_cleaned["DateTime"] = pd.to_datetime(df_cleaned["DateTime"])
df_cleaned.set_index("DateTime", inplace=True)

nan_values = df_cleaned["posX"] == (" -")

display(nan_values)

display(nan_values.loc[lambda x: x == True])

group_id = (nan_values != nan_values.shift()).cumsum()
groups = df_cleaned.groupby(group_id)

display(group_id)

to_remove = []
removed_frames = 0

for _, group in groups:
    val = nan_values.loc[group.index[0]]
    if isinstance(val, pd.Series):   
        val = val.iloc[0]  # Sicherstellen, dass es ein einzelner Wert ist
    if val:  # Nur Null-ähnliche Gruppen        
        to_remove.append(group.index)
        removed_frames += len(group)

display(to_remove)

remove_idx =  pd.Index(np.concatenate(to_remove)) if to_remove else pd.Index([])
df_cleaned2 = df_cleaned.drop(index=remove_idx)

print("Anzahl der entfernten Nullwerte:", removed_frames)

exp_data = df_cleaned2[(df_cleaned2["Proband"] == proband) & (df_cleaned2["Task"] == task) & (df_cleaned2["Block"] == block) & (df_cleaned2["Trial"] == trial)].reset_index(drop=True)

display(exp_data)

,Unnamed: 0,SampleIdx_global,SampleIdx_Proband,DateTime,State,mappingMethod,Task,TargetLayers,NumLayers,Trial,...,posZ,TimeStamp,iteractionType,currentLayer,Proband,shifted,Result,Block,Target,Target_Relative
0,0,135,135,2021-05-10T12:13:56.584Z,VIEW,direct,1,"['4', '5', '3', '1', '2']",6,0,...,NaN,NaN,NaN,NaN,1,1.0,START,1,4,0.666667
1,1,136,136,2021-05-10T12:13:56.602Z,INTERACTION,direct,1,"['4', '5', '3', '1', '2']",6,0,...,-,-,-,2,1,1.0,COMPLETED,1,4,0.666667
2,2,137,137,2021-05-10T12:13:56.633Z,INTERACTION,direct,1,"['4', '5', '3', '1', '2']",6,0,...,-,-,-,2,1,1.0,COMPLETED,1,4,0.666667
3,3,138,138,2021-05-10T12:13:56.676Z,INTERACTION,direct,1,"['4', '5', '3', '1', '2']",6,0,...,-,-,-,-,1,1.0,COMPLETED,1,4,0.666667
4,4,139,139,2021-05-10T12:13:56.697Z,INTERACTION,direct,1,"['4', '5', '3', '1', '2']",6,0,...,-,-,-,-,1,1.0,COMPLETED,1,4,0.666667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1535230,1535230,2360983,90204,2021-06-30T12:37:15.245Z,INTERACTION,widening,18,"['4', '5', '2', '3', '1']",6,4,...,-0.9999999,637606606352421800,1,6,24,54.0,FAILED,3,1,0.166667
1535231,1535231,2360984,90205,2021-06-30T12:37:15.291Z,INTERACTION,widening,18,"['4', '5', '2', '3', '1']",6,4,...,-0.9999999,637606606352861800,1,6,24,54.0,FAILED,3,1,0.166667
1535232,1535232,2360985,90206,2021-06-30T12:37:15.323Z,INTERACTION,widening,18,"['4', '5', '2', '3', '1']",6,4,...,-0.9999999,637606606353161900,1,6,24,54.0,FAILED,3,1,0.166667
1535233,1535233,2360986,90207,2021-06-30T12:37:15.361Z,INTERACTION,widening,18,"['4', '5', '2', '3', '1']",6,4,...,-0.9999999,637606606353551700,1,6,24,54.0,FAILED,3,1,0.166667


,Unnamed: 0,SampleIdx_global,SampleIdx_Proband,DateTime,State,mappingMethod,Task,TargetLayers,NumLayers,Trial,...,posZ,TimeStamp,iteractionType,currentLayer,Proband,shifted,Result,Block,Target,Target_Relative
0,1189494,1855631,28737,2021-06-21T16:48:09.752Z,SUBTASK_STATE,widening,4,"['17', '5', '1', '9', '14']",18,3,...,NaN,NaN,NaN,NaN,18,22.0,START,2,9,0.5
1,1189495,1855632,28738,2021-06-21T16:48:09.784Z,INTERACTION,widening,4,"['17', '5', '1', '9', '14']",18,3,...,-0.769736648,637598980897781800,1,13,18,22.0,COMPLETED,2,9,0.5
2,1189496,1855633,28739,2021-06-21T16:48:09.825Z,INTERACTION,widening,4,"['17', '5', '1', '9', '14']",18,3,...,-0.8210525,637598980898231700,1,15,18,22.0,COMPLETED,2,9,0.5
3,1189497,1855634,28740,2021-06-21T16:48:09.868Z,INTERACTION,widening,4,"['17', '5', '1', '9', '14']",18,3,...,-0.8210525,637598980898619400,1,16,18,22.0,COMPLETED,2,9,0.5
4,1189498,1855635,28741,2021-06-21T16:48:09.900Z,INTERACTION,widening,4,"['17', '5', '1', '9', '14']",18,3,...,-0.8565788,637598980898949400,1,16,18,22.0,COMPLETED,2,9,0.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9554,1199048,1865185,38291,2021-06-21T16:54:14.981Z,INTERACTION,widening,4,"['17', '5', '1', '9', '14']",18,3,...,-0.335526258,637598984549783700,1,9,18,22.0,COMPLETED,2,9,0.5
9555,1199049,1865186,38292,2021-06-21T16:54:15.023Z,INTERACTION,widening,4,"['17', '5', '1', '9', '14']",18,3,...,-0.339473575,637598984550213600,1,9,18,22.0,COMPLETED,2,9,0.5
9556,1199050,1865187,38293,2021-06-21T16:54:15.063Z,INTERACTION,widening,4,"['17', '5', '1', '9', '14']",18,3,...,-0.339473575,637598984550603800,1,9,18,22.0,COMPLETED,2,9,0.5
9557,1199051,1865188,38294,2021-06-21T16:54:15.092Z,INTERACTION,widening,4,"['17', '5', '1', '9', '14']",18,3,...,-0.339473575,637598984550873600,1,9,18,22.0,COMPLETED,2,9,0.5


DateTime
2021-05-10 12:13:56.584000+00:00    False
2021-05-10 12:13:56.602000+00:00     True
2021-05-10 12:13:56.633000+00:00     True
2021-05-10 12:13:56.676000+00:00     True
2021-05-10 12:13:56.697000+00:00     True
                                    ...  
2021-06-30 12:37:15.245000+00:00    False
2021-06-30 12:37:15.291000+00:00    False
2021-06-30 12:37:15.323000+00:00    False
2021-06-30 12:37:15.361000+00:00    False
2021-06-30 12:37:15.381000+00:00    False
Name: posX, Length: 1535235, dtype: bool

DateTime
2021-05-10 12:13:56.602000+00:00    True
2021-05-10 12:13:56.633000+00:00    True
2021-05-10 12:13:56.676000+00:00    True
2021-05-10 12:13:56.697000+00:00    True
2021-05-10 12:13:56.746000+00:00    True
                                    ... 
2021-06-30 12:37:11.081000+00:00    True
2021-06-30 12:37:11.119000+00:00    True
2021-06-30 12:37:11.164000+00:00    True
2021-06-30 12:37:11.205000+00:00    True
2021-06-30 12:37:11.240000+00:00    True
Name: posX, Length: 400527, dtype: bool

DateTime
2021-05-10 12:13:56.584000+00:00        1
2021-05-10 12:13:56.602000+00:00        2
2021-05-10 12:13:56.633000+00:00        2
2021-05-10 12:13:56.676000+00:00        2
2021-05-10 12:13:56.697000+00:00        2
                                    ...  
2021-06-30 12:37:15.245000+00:00    16365
2021-06-30 12:37:15.291000+00:00    16365
2021-06-30 12:37:15.323000+00:00    16365
2021-06-30 12:37:15.361000+00:00    16365
2021-06-30 12:37:15.381000+00:00    16365
Name: posX, Length: 1535235, dtype: int32

[DatetimeIndex(['2021-05-10 12:13:56.602000+00:00',
                '2021-05-10 12:13:56.633000+00:00',
                '2021-05-10 12:13:56.676000+00:00',
                '2021-05-10 12:13:56.697000+00:00',
                '2021-05-10 12:13:56.746000+00:00',
                '2021-05-10 12:13:56.800000+00:00',
                '2021-05-10 12:13:56.842000+00:00',
                '2021-05-10 12:13:56.879000+00:00',
                '2021-05-10 12:13:56.914000+00:00',
                '2021-05-10 12:13:56.960000+00:00',
                '2021-05-10 12:13:56.995000+00:00',
                '2021-05-10 12:13:57.037000+00:00',
                '2021-05-10 12:13:57.095000+00:00',
                '2021-05-10 12:13:57.126000+00:00',
                '2021-05-10 12:13:57.161000+00:00',
                '2021-05-10 12:13:57.193000+00:00',
                '2021-05-10 12:13:57.238000+00:00',
                '2021-05-10 12:13:57.290000+00:00',
                '2021-05-10 12:13:57.333000+00:00',
            

Anzahl der entfernten Nullwerte: 401213


,Unnamed: 0,SampleIdx_global,SampleIdx_Proband,State,mappingMethod,Task,TargetLayers,NumLayers,Trial,posX,...,posZ,TimeStamp,iteractionType,currentLayer,Proband,shifted,Result,Block,Target,Target_Relative
0,1189494,1855631,28737,SUBTASK_STATE,widening,4,"['17', '5', '1', '9', '14']",18,3,START,...,NaN,NaN,NaN,NaN,18,22.0,START,2,9,0.5
1,1189495,1855632,28738,INTERACTION,widening,4,"['17', '5', '1', '9', '14']",18,3,0.195627853,...,-0.769736648,637598980897781800,1,13,18,22.0,COMPLETED,2,9,0.5
2,1189496,1855633,28739,INTERACTION,widening,4,"['17', '5', '1', '9', '14']",18,3,0.1848301,...,-0.8210525,637598980898231700,1,15,18,22.0,COMPLETED,2,9,0.5
3,1189497,1855634,28740,INTERACTION,widening,4,"['17', '5', '1', '9', '14']",18,3,0.1848301,...,-0.8210525,637598980898619400,1,16,18,22.0,COMPLETED,2,9,0.5
4,1189498,1855635,28741,INTERACTION,widening,4,"['17', '5', '1', '9', '14']",18,3,0.17597948,...,-0.8565788,637598980898949400,1,16,18,22.0,COMPLETED,2,9,0.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
198,1199048,1865185,38291,INTERACTION,widening,4,"['17', '5', '1', '9', '14']",18,3,0.3192707,...,-0.335526258,637598984549783700,1,9,18,22.0,COMPLETED,2,9,0.5
199,1199049,1865186,38292,INTERACTION,widening,4,"['17', '5', '1', '9', '14']",18,3,0.323519021,...,-0.339473575,637598984550213600,1,9,18,22.0,COMPLETED,2,9,0.5
200,1199050,1865187,38293,INTERACTION,widening,4,"['17', '5', '1', '9', '14']",18,3,0.323519021,...,-0.339473575,637598984550603800,1,9,18,22.0,COMPLETED,2,9,0.5
201,1199051,1865188,38294,INTERACTION,widening,4,"['17', '5', '1', '9', '14']",18,3,0.323519021,...,-0.339473575,637598984550873600,1,9,18,22.0,COMPLETED,2,9,0.5
